# 08 - Integrasi ECG ESP32 melalui USB Serial

Notebook membaca ECG dari ESP32 melalui **COM3 pada 115200 baud** dan menjalankan klasifikasi **CHF vs NSR**. Data bertimestamp diresampling ke 128 Hz seperti dataset training. Prediksi diperbarui setiap 60 detik. Sebelum terkumpul satu rekaman penuh (65.536 sampel/sekitar 8,5 menit), hasil diberi status **sementara**; hasil final menggunakan semua window dalam rekaman seperti evaluasi per rekaman. Grafik 10 detik terakhir ditampilkan bersama hasil.

Model ini hanya mengenali CHF dan NSR, bukan aritmia/ARR. Hasilnya eksperimen, bukan diagnosis medis. Firmware ESP32 harus mengirim `micros,ADC` atau `micros,LEAD_OFF`; tutup Serial Monitor/Plotter Arduino IDE sebelum menjalankan capture.


In [ ]:
from pathlib import Path
from collections import deque
from datetime import datetime
import json
import pickle
import time

import numpy as np
import pywt
import serial
import tensorflow as tf
from serial.tools import list_ports
import matplotlib.pyplot as plt
from IPython.display import display

# Cari folder project dari direktori notebook, project, atau workspace induk.
current_dir = Path.cwd()
project_candidates = [current_dir, *current_dir.parents]
project_candidates.extend(path for path in current_dir.iterdir() if path.is_dir())
ROOT = next(
    (path.resolve() for path in project_candidates
     if (path / "notebooks").is_dir() and (path / "data").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Folder project tidak ditemukan; jalankan notebook dari folder project atau notebooks.")

DATA_DIR = ROOT / "data"
WORK_DIR = DATA_DIR / "processed"
MODEL_DIR = ROOT / "06_model_chf_nsr"
SERIAL_PORT = "COM3"
BAUD_RATE = 115200
SERIAL_FS = 128
WINDOW_LENGTH = 3136
HOP_LENGTH = 784
RECORD_LENGTH = 65536
PREDICTION_INTERVAL_SECONDS = 60
FULL_RECORD_SECONDS = RECORD_LENGTH / SERIAL_FS
MAX_SAMPLE_GAP_SECONDS = 0.1
RAW_BUFFER_CAPACITY = RECORD_LENGTH + 2 * SERIAL_FS
PLOT_WINDOW_SECONDS = 10
PLOT_UPDATE_SECONDS = PREDICTION_INTERVAL_SECONDS
PLOT_WINDOW_SAMPLES = SERIAL_FS * PLOT_WINDOW_SECONDS
RUN_LIVE_CAPTURE = True

print("Project:", ROOT)
print("PySerial:", serial.__version__)
print("TensorFlow:", tf.__version__)
print("Serial port:", SERIAL_PORT, "at", BAUD_RATE, "baud")
print("Klasifikasi diperbarui setiap", PREDICTION_INTERVAL_SECONDS, "detik")
print("Durasi rekaman penuh:", FULL_RECORD_SECONDS, "detik")

Project: C:\Users\LENOVO\Downloads\BELAJAR\SEM 5\BIOMED\ECG_2D_CNN_Project_Dataset_AutoDownload\ECG_2D_CNN_Project
PySerial: 3.5
TensorFlow: 2.21.0
Serial port: COM3 at 115200 baud
Klasifikasi diperbarui setiap 60 detik
Durasi rekaman penuh: 512.0 detik


In [ ]:
MODEL_PATH = MODEL_DIR / "best_model.keras"
ENCODER_PATH = MODEL_DIR / "label_encoder.pkl"
WAVELET_CONFIG_PATH = WORK_DIR / "03_denoised" / "pso_wavelet_config.json"

for required_file in (MODEL_PATH, ENCODER_PATH, WAVELET_CONFIG_PATH):
    if not required_file.is_file():
        raise FileNotFoundError(f"File integrasi tidak ditemukan: {required_file}")

with ENCODER_PATH.open("rb") as file:
    le = pickle.load(file)
model = tf.keras.models.load_model(MODEL_PATH)
with WAVELET_CONFIG_PATH.open("r", encoding="utf-8") as file:
    pso_cfg = json.load(file)
if list(le.classes_) != ["CHF", "NSR"]:
    raise ValueError(f"Encoder harus berisi kelas CHF dan NSR; ditemukan {list(le.classes_)}")
if tuple(model.input_shape[1:]) != (28, 28, 4) or model.output_shape[-1] != 2:
    raise ValueError(f"Bentuk model bukan model biner 4-kanal: input={model.input_shape}, output={model.output_shape}")

def sure_threshold(detail, sigma):
    values = np.asarray(detail, dtype=np.float64).ravel() / (sigma + 1e-12)
    n = values.size
    if n == 0:
        return 0.0
    absolute_values = np.sort(np.abs(values))
    counts = np.arange(1, n + 1, dtype=np.float64)
    cumulative_squares = np.cumsum(absolute_values ** 2)
    risks = (n - 2 * counts + cumulative_squares
             + (n - counts) * absolute_values ** 2)
    return float(np.sqrt(max(0.0, np.min(risks))) * sigma)

def wavelet_denoise(signal, wavelet, level, threshold_type):
    coefficients = pywt.wavedec(signal, wavelet, level=level, mode="symmetric")
    finest_detail = coefficients[-1]
    sigma = np.median(np.abs(finest_detail - np.median(finest_detail))) / 0.6745
    sigma = max(float(sigma), 1e-8)
    filtered = [coefficients[0]]
    for detail in coefficients[1:]:
        threshold = sure_threshold(detail, sigma)
        filtered.append(pywt.threshold(detail, threshold, mode=threshold_type))
    return pywt.waverec(filtered, wavelet, mode="symmetric")[:len(signal)].astype(np.float32)

def normalize_minmax(signal):
    minimum, maximum = np.min(signal), np.max(signal)
    if maximum - minimum < 1e-12:
        return np.zeros_like(signal, dtype=np.float32)
    return (2.0 * (signal - minimum) / (maximum - minimum) - 1.0).astype(np.float32)

def prepare_denoised_window(denoised_window):
    signal = np.asarray(denoised_window, dtype=np.float32)
    if signal.shape != (WINDOW_LENGTH,):
        raise ValueError(f"Window harus berisi {WINDOW_LENGTH} sampel, diterima {signal.shape}.")
    image = normalize_minmax(signal).reshape(56, 56)
    ll, (lh, hl, hh) = pywt.dwt2(image, "haar", mode="periodization")
    return np.stack([ll, lh, hl, hh], axis=-1).astype(np.float32)

def denoise_live_signal(signal):
    wavelet = pso_cfg["wavelet"]
    level = min(int(pso_cfg["level"]),
                pywt.dwt_max_level(len(signal), pywt.Wavelet(wavelet).dec_len))
    return wavelet_denoise(signal, wavelet, level, pso_cfg["threshold_type"])

def resample_timed_capture(sample_times, sample_values):
    times = np.asarray(sample_times, dtype=np.float64)
    values = np.asarray(sample_values, dtype=np.float32)
    if times.ndim != 1 or values.shape != times.shape or len(times) < 2:
        raise ValueError("Belum cukup sampel bertimestamp untuk resampling.")
    intervals = np.diff(times)
    if np.any(intervals <= 0):
        raise ValueError("Timestamp serial tidak meningkat.")
    if np.max(intervals) > MAX_SAMPLE_GAP_SECONDS:
        raise ValueError("Jeda sampel terlalu besar; mulai ulang pengambilan data.")
    duration = times[-1] - times[0]
    full_record = duration >= (RECORD_LENGTH - 1) / SERIAL_FS
    if full_record:
        count = RECORD_LENGTH
        start_time = times[-1] - (count - 1) / SERIAL_FS
    else:
        count = min(RECORD_LENGTH, int(np.floor(duration * SERIAL_FS)) + 1)
        start_time = times[0]
    if count < WINDOW_LENGTH:
        raise ValueError("Belum terkumpul satu window model.")
    target_times = start_time + np.arange(count, dtype=np.float64) / SERIAL_FS
    signal = np.interp(target_times, times, values).astype(np.float32)
    return signal, full_record

def predict_live_capture(raw_capture):
    signal = np.asarray(raw_capture, dtype=np.float32)
    if signal.ndim != 1 or len(signal) < WINDOW_LENGTH:
        raise ValueError(f"Capture minimal {WINDOW_LENGTH} sampel.")
    signal = signal[-RECORD_LENGTH:]
    denoised = denoise_live_signal(signal)
    starts = range(0, len(denoised) - WINDOW_LENGTH + 1, HOP_LENGTH)
    images = np.stack([
        prepare_denoised_window(denoised[start:start + WINDOW_LENGTH])
        for start in starts
    ])
    window_probabilities = model.predict(images, batch_size=128, verbose=0)
    probabilities = np.mean(window_probabilities, axis=0)
    predicted_class = le.classes_[int(np.argmax(probabilities))]
    return predicted_class, probabilities, len(window_probabilities)

def send_classification_to_esp32(serial_connection, label, probability):
    if label not in ("WAITING", "CHF", "NSR"):
        raise ValueError(f"Label serial tidak dikenal: {label}")
    probability = float(probability)
    if not np.isfinite(probability) or not 0.0 <= probability <= 1.0:
        raise ValueError(f"Probabilitas harus dalam rentang 0..1: {probability}")
    serial_connection.write(f"{label},{probability:.3f}\n".encode("ascii"))
    serial_connection.flush()

print("Binary model loaded:", MODEL_PATH.name)
print("Input shape:", model.input_shape)
print("Classes:", list(le.classes_))


Binary model loaded: best_model.keras
Input shape: (None, 28, 28, 4)
Classes: [np.str_('CHF'), np.str_('NSR')]


In [ ]:
if not RUN_LIVE_CAPTURE:
    print("Live capture nonaktif. Ubah RUN_LIVE_CAPTURE=True setelah ESP32 siap.")
else:
    available_ports = {port.device.upper() for port in list_ports.comports()}
    if SERIAL_PORT.upper() not in available_ports:
        found = ", ".join(sorted(available_ports)) or "tidak ada"
        print(
            f"Live capture tidak dimulai: {SERIAL_PORT} tidak terdeteksi. "
            f"Port yang ditemukan: {found}. Periksa kabel USB, driver, dan SERIAL_PORT."
        )
    else:
        try:
            ser = serial.Serial(
                port=SERIAL_PORT, baudrate=BAUD_RATE, timeout=1, write_timeout=1
            )
        except serial.SerialException as exc:
            print(
                f"Live capture tidak dimulai: tidak dapat membuka {SERIAL_PORT}. "
                "Tutup Serial Monitor/Plotter Arduino IDE atau aplikasi lain yang memakai port. "
                f"Detail: {exc}"
            )
        else:
            sample_times = deque(maxlen=RAW_BUFFER_CAPACITY)
            sample_values = deque(maxlen=RAW_BUFFER_CAPACITY)
            signal_buffer = deque(maxlen=PLOT_WINDOW_SAMPLES)
            samples_received = 0
            invalid_lines = 0
            prediction_no = 0
            previous_raw_micros = None
            micros_offset = 0
            previous_sample_time = None
            last_sample_host_time = None
            last_prediction_time = time.monotonic()
            last_wait_notice = last_prediction_time
            lead_status = "menunggu data"
            fig = None
            plot_display = None

            print(
                f"Terhubung ke {SERIAL_PORT} pada {BAUD_RATE} baud. "
                f"Prediksi diperbarui setiap {PREDICTION_INTERVAL_SECONDS} detik; "
                f"hasil final memerlukan sekitar {FULL_RECORD_SECONDS / 60:.1f} menit data valid. "
                "Tekan Interrupt untuk berhenti."
            )
            try:
                time.sleep(2)
                ser.reset_input_buffer()
                last_prediction_time = time.monotonic()
                last_wait_notice = last_prediction_time

                fig, ax = plt.subplots(figsize=(12, 3))
                signal_line, = ax.plot([], [], linewidth=0.8)
                ax.set_xlim(0, PLOT_WINDOW_SECONDS)
                ax.set_ylim(0, 4095)
                ax.set_xlabel("Waktu (detik)")
                ax.set_ylabel("Nilai ADC")
                ax.set_title(f"Sinyal ECG live dari {SERIAL_PORT} - {lead_status}")
                ax.grid(True, alpha=0.3)
                fig.tight_layout()

                while True:
                    raw_line = ser.readline()
                    if raw_line:
                        line = raw_line.decode("ascii", errors="ignore").strip()
                        parts = line.split(",", 1)
                        if len(parts) == 1:
                            try:
                                value = float(parts[0].strip())
                            except ValueError:
                                value = None
                            if value is not None and np.isfinite(value) and 0 <= value <= 4095:
                                sample_time = (
                                    0.0 if previous_sample_time is None
                                    else previous_sample_time + 1.0 / SERIAL_FS
                                )
                                sample_times.append(sample_time)
                                sample_values.append(value)
                                signal_buffer.append(value)
                                previous_sample_time = sample_time
                                last_sample_host_time = time.monotonic()
                                samples_received += 1
                                lead_status = "sinyal diterima (ADC tunggal)"
                            else:
                                invalid_lines += 1
                        else:
                            try:
                                raw_micros = int(parts[0])
                            except ValueError:
                                raw_micros = -1
                            sample_text = parts[1].strip()

                            if not 0 <= raw_micros <= 0xFFFFFFFF:
                                invalid_lines += 1
                            else:
                                if previous_raw_micros is not None and raw_micros < previous_raw_micros:
                                    if previous_raw_micros - raw_micros > 0x80000000:
                                        micros_offset += 0x100000000
                                    else:
                                        sample_times.clear()
                                        sample_values.clear()
                                        signal_buffer.clear()
                                        micros_offset = 0
                                        previous_sample_time = None
                                        last_sample_host_time = None
                                        last_prediction_time = time.monotonic()
                                previous_raw_micros = raw_micros
                                sample_time = (micros_offset + raw_micros) / 1_000_000.0

                                if sample_text == "LEAD_OFF":
                                    if not lead_status.startswith("LEAD_OFF"):
                                        send_classification_to_esp32(ser, "WAITING", 0.0)
                                    sample_times.clear()
                                    sample_values.clear()
                                    signal_buffer.clear()
                                    previous_sample_time = None
                                    last_sample_host_time = None
                                    last_prediction_time = time.monotonic()
                                    lead_status = "LEAD_OFF - periksa elektroda"
                                else:
                                    try:
                                        value = float(sample_text)
                                    except ValueError:
                                        value = None
                                    if value is not None and np.isfinite(value) and 0 <= value <= 4095:
                                        if (
                                            previous_sample_time is not None
                                            and sample_time - previous_sample_time > MAX_SAMPLE_GAP_SECONDS
                                        ):
                                            sample_times.clear()
                                            sample_values.clear()
                                            signal_buffer.clear()
                                            last_sample_host_time = None
                                            last_prediction_time = time.monotonic()
                                            lead_status = "jeda sampel - buffer diulang"
                                        sample_times.append(sample_time)
                                        sample_values.append(value)
                                        signal_buffer.append(value)
                                        previous_sample_time = sample_time
                                        last_sample_host_time = time.monotonic()
                                        samples_received += 1
                                        lead_status = "sinyal diterima"

                    now = time.monotonic()
                    if (
                        last_sample_host_time is not None
                        and now - last_sample_host_time > MAX_SAMPLE_GAP_SECONDS
                    ):
                        sample_times.clear()
                        sample_values.clear()
                        signal_buffer.clear()
                        previous_sample_time = None
                        last_sample_host_time = None
                        last_prediction_time = now
                        lead_status = "sinyal terhenti - buffer diulang"
                    if (
                        len(sample_times) >= 2
                        and last_sample_host_time is not None
                        and now - last_sample_host_time <= MAX_SAMPLE_GAP_SECONDS
                        and now - last_prediction_time >= PREDICTION_INTERVAL_SECONDS
                    ):
                        try:
                            capture, full_record = resample_timed_capture(
                                list(sample_times), list(sample_values)
                            )
                            predicted_class, probabilities, window_count = predict_live_capture(capture)
                        except ValueError as exc:
                            print(f"Prediksi ditunda: {exc}")
                        else:
                            probability_by_class = {
                                str(class_name): float(probability)
                                for class_name, probability in zip(le.classes_, probabilities)
                            }
                            confidence = float(np.max(probabilities))
                            prediction_no += 1
                            stage = "FINAL" if full_record else "SEMENTARA"
                            print(f"\nHasil #{prediction_no} ({stage}) - {datetime.now().astimezone():%Y-%m-%d %H:%M:%S}")
                            print("Prediksi:", predicted_class if full_record else f"sementara: {predicted_class}")
                            print(f"Window dianalisis: {window_count}; durasi: {len(capture) / SERIAL_FS:.1f} detik")
                            print(
                                "Probabilitas rata-rata: "
                                + " | ".join(
                                    f"{name}: {probability_by_class[name]:.3f}"
                                    for name in ("CHF", "NSR")
                                )
                            )
                            if full_record:
                                try:
                                    send_classification_to_esp32(ser, predicted_class, confidence)
                                    print(f"Hasil final dikirim ke OLED: {predicted_class},{confidence:.3f}")
                                except serial.SerialException as exc:
                                    print(f"Gagal mengirim klasifikasi ke ESP32: {exc}")
                            else:

                            y_values = np.asarray(signal_buffer, dtype=np.float32)
                            if y_values.size:
                                x_values = np.arange(y_values.size, dtype=np.float32) / SERIAL_FS
                                signal_line.set_data(x_values, y_values)
                                plot_end = float(x_values[-1])
                                ax.set_xlim(
                                    max(0.0, plot_end - PLOT_WINDOW_SECONDS),
                                    max(float(PLOT_WINDOW_SECONDS), plot_end),
                                )
                                y_min = float(np.min(y_values))
                                y_max = float(np.max(y_values))
                                y_padding = max((y_max - y_min) * 0.1, 1.0)
                                ax.set_ylim(max(0.0, y_min - y_padding), min(4095.0, y_max + y_padding))
                                ax.set_title(f"Sinyal ECG live dari {SERIAL_PORT} - {lead_status}")
                                if plot_display is None:
                                    plot_display = display(fig, display_id=True)
                                else:
                                    plot_display.update(fig)
                            last_prediction_time = time.monotonic()

                    if now - last_wait_notice >= 10:
                        if not sample_times:
                            print("Belum menerima data timestamp,ADC. Firmware harus mengirim `micros,ADC` atau `micros,LEAD_OFF`.")
                        else:
                            duration = sample_times[-1] - sample_times[0]
                            print(
                                f"Data valid: {duration:.1f}/{FULL_RECORD_SECONDS:.1f} detik menuju rekaman penuh; "
                                f"sampel diterima total: {samples_received}."
                            )
                        if invalid_lines:
                            print(f"Baris serial dengan format tidak valid: {invalid_lines}.")
                        last_wait_notice = now
            except KeyboardInterrupt:
                print("Pembacaan dihentikan.")
            except serial.SerialException as exc:
                print(f"Koneksi serial terputus: {exc}")
            finally:
                if ser.is_open:
                    ser.close()
                print(f"Port {SERIAL_PORT} ditutup.")
                if fig is not None:
                    plt.close(fig)


IndentationError: expected an indented block after 'else' statement on line 192 (481482039.py, line 194)